## Load Metadata - IBM Db2 LUW
Discovers tables from a Db2 schema via `remote_query()` on `SYSCAT.TABLES` and merges them into the control table.

**No foreign catalog required** — only a Unity Catalog JDBC connection.
All ingestion operations use `use_remote_query = true`.

### Steps
1. Edit the **Parameters** cell with your values and run it
2. Edit **Per-Table Configuration Overrides** to set partitioning, clustering, and watermark config
3. Run **Discover Db2 Tables** to preview `control_src` before merging
4. Run **Merge into Control Table** to register tasks

In [0]:
-- Edit these values, then run this cell before running the rest of the notebook.
DECLARE OR REPLACE VARIABLE src_connection  STRING DEFAULT 'db2_cloud_connection';
DECLARE OR REPLACE VARIABLE src_database    STRING DEFAULT 'MYDB';
DECLARE OR REPLACE VARIABLE src_schema      STRING DEFAULT 'MYSCHEMA';
DECLARE OR REPLACE VARIABLE task_collection STRING DEFAULT 'db2_prod';
DECLARE OR REPLACE VARIABLE sink_catalog    STRING DEFAULT 'lakefed_ingest';
DECLARE OR REPLACE VARIABLE sink_schema     STRING DEFAULT 'db2_prod';
DECLARE OR REPLACE VARIABLE ctrl_catalog    STRING DEFAULT 'lakefed_ingest';
DECLARE OR REPLACE VARIABLE ctrl_schema     STRING DEFAULT 'default';
DECLARE OR REPLACE VARIABLE ctrl_table      STRING DEFAULT 'control';

In [0]:
-- Define per-table overrides for partitioning, clustering, and incremental loads.
-- Tables NOT listed here default to: full load, non-partitioned, no clustering.

CREATE OR REPLACE TEMP VIEW table_details AS
SELECT
  col1                           AS table_name,
  IF(col2 = 'y', true, false)    AS load_partitioned,
  col3                            AS partition_col,
  SPLIT(col4, ',')                AS sink_cluster_cols,
  SPLIT(col5, ',')                AS primary_key,
  col6                            AS watermark_col_name,
  col7                            AS watermark_col_type,
  col8                            AS watermark_col_start_value
FROM (
  VALUES
    -- table_name      load_partitioned  partition_col  sink_cluster_cols  primary_key      watermark_col_name  watermark_col_type  watermark_col_start_value
    ('orders',         'n',              null,    null,      'order_id',      'updated_at',       'timestamp',        "1000-01-01 00:00:00"),
    ('customers',      'n',              null,          null,              'customer_id',   'updated_at',       'timestamp',        "1000-01-01 00:00:00")
);

In [0]:
-- Step 1: Discover tables from Db2 SYSCAT.TABLES via remote_query.
-- EXECUTE IMMEDIATE is required because src_schema is a session variable.
-- Note: database is embedded in the JDBC URL and cannot be passed as an option externally.
DECLARE OR REPLACE qry STRING;

SET VAR qry =
  'CREATE OR REPLACE TEMP VIEW db2_tables AS '
  || 'SELECT * FROM remote_query('
  || chr(39) || src_connection || chr(39)
  || ', query => ' || chr(39)
  ||   'SELECT TABSCHEMA, TABNAME FROM SYSCAT.TABLES'
  ||   ' WHERE TABSCHEMA = UPPER(' || chr(92)||chr(39) || src_schema || chr(92)||chr(39) || ')'
  ||   ' AND TYPE = ' || chr(92)||chr(39) || 'T' || chr(92)||chr(39)
  || chr(39) || ')';

EXECUTE IMMEDIATE qry;

-- Step 2: Join discovered tables with per-table overrides to build control_src.
USE CATALOG identifier(ctrl_catalog);
USE SCHEMA identifier(ctrl_schema);

CREATE OR REPLACE TEMP VIEW control_src AS
SELECT
  'lakefed_ingest'                                                  AS job_name,
  task_collection                                                    AS task_collection,
  'db2_luw'                                                         AS src_type,
  src_connection                                                     AS src_connection,
  src_database                                                       AS src_database,
  NULL                                                              AS src_catalog,
  LOWER(t.TABSCHEMA)                                                AS src_schema,
  LOWER(t.TABNAME)                                                  AS src_table,
  sink_catalog                                                       AS sink_catalog,
  sink_schema                                                        AS sink_schema,
  LOWER(t.TABNAME)                                                  AS sink_table,
  false                                                             AS enable_iceberg_reads,
  COALESCE(d.primary_key, ARRAY())                                  AS primary_key,
  COALESCE(d.sink_cluster_cols, ARRAY())                            AS sink_cluster_cols,
  CASE WHEN d.watermark_col_name IS NOT NULL THEN 'incremental'
       ELSE 'full' END                                             AS load_type,
  COALESCE(d.load_partitioned, false)                               AS load_partitioned,
  '*'                                                               AS select_list,
  d.watermark_col_name,
  d.watermark_col_type,
  d.watermark_col_start_value,
  CASE WHEN d.load_partitioned THEN d.partition_col ELSE NULL END   AS partition_col,
  CASE WHEN d.load_partitioned THEN 512 ELSE NULL END               AS partition_size_mb,
  true                                                              AS use_remote_query,
  true                                                              AS task_enabled
FROM db2_tables t
LEFT JOIN table_details d ON LOWER(t.TABNAME) = d.table_name;

SELECT * FROM control_src;

In [0]:
-- Merges discovered tables into the control table.
-- Matched on src_type + src_schema + src_table so re-running is safe (updates existing rows).
-- WARNING: this overwrites existing watermarks if rows already exist.
-- To preserve watermarks, add AND t.watermark_col_start_value IS NOT NULL to the WHEN MATCHED condition.

MERGE WITH SCHEMA EVOLUTION INTO identifier(ctrl_catalog || '.' || ctrl_schema || '.' || ctrl_table) AS t
USING control_src AS s
ON  t.src_type   = s.src_type
AND t.src_schema = s.src_schema
AND t.src_table  = s.src_table
WHEN MATCHED     THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;